# Master updater for both codebases

This notebook is the shared updater for the two repos inside this workspace:
- `brojogopalsapui.github.io`
- `AiSecurityResearch`

It supports two workflows:
1. **Weekly post sync from one DOCX** across both codebases and their `papers_articles` mirrors.
2. **Static article/page sync** from one source site root to every other site root.

## Folder logic

This notebook lives in `common_internal/`, which sits next to both repos.

Expected workspace shape:

```text
Git_workspace/
  common_internal/
    master_update_both_codebases_from_docx.ipynb
    incoming_docx/
    article_sync_manifest.txt
  brojogopalsapui.github.io/
  AiSecurityResearch/
```

For weekly posts, put the filled DOCX into `common_internal/incoming_docx/`.

In [ ]:
from pathlib import Path
import sys

THIS_DIR = Path.cwd().resolve()
if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from site_sync_tools import (
    discover_workspace_root,
    list_site_roots,
    pick_latest_docx,
    sync_weekly_post_to_many_sites,
    read_manifest,
    sync_article_manifest_to_sites,
)

WORKSPACE_ROOT = discover_workspace_root(THIS_DIR)
REPO_NAMES = ["brojogopalsapui.github.io", "AiSecurityResearch"]
INCLUDE_PAPERS_ARTICLES_MIRRORS = True
SITE_ROOTS = list_site_roots(WORKSPACE_ROOT, REPO_NAMES, include_papers_articles=INCLUDE_PAPERS_ARTICLES_MIRRORS)

print("Workspace root:", WORKSPACE_ROOT)
print("Site roots that will be handled:")
for site_root in SITE_ROOTS:
    print(" -", site_root)

In [ ]:
# =========================
# WORKFLOW SWITCHES
# =========================
RUN_WEEKLY_POST_SYNC = True
RUN_ARTICLE_SYNC = False

# =========================
# WEEKLY POST DOCX SETTINGS
# =========================
DOCX_INPUT_DIR = THIS_DIR / "incoming_docx"
AUTO_PICK_LATEST_DOCX = True
MANUAL_DOCX_PATH = DOCX_INPUT_DIR / "2026-04-secure-agent-memory.docx"
HOME_SLIDER_MAX_CARDS = 6
COPY_DOCX_INTO_EACH_SITE = True
REPLACE_DUPLICATE_POST_ID = True
UPDATE_HOME_FLOATING = True
UPDATE_HOME_SLIDER = True

# =========================
# STATIC ARTICLE/PAGE SYNC SETTINGS
# =========================
ARTICLE_SOURCE_REPO_NAME = "brojogopalsapui.github.io"
ARTICLE_SOURCE_SITE_RELATIVE = "."   # use "." for repo root, or "papers_articles" for that mirror
ARTICLE_MANIFEST_PATH = THIS_DIR / "article_sync_manifest.txt"

In [ ]:
selected_docx = pick_latest_docx(DOCX_INPUT_DIR) if AUTO_PICK_LATEST_DOCX else MANUAL_DOCX_PATH
print("Selected DOCX:", selected_docx)

In [ ]:
if RUN_WEEKLY_POST_SYNC:
    weekly_results = sync_weekly_post_to_many_sites(
        site_roots=SITE_ROOTS,
        docx_path=selected_docx,
        home_slider_max_cards=HOME_SLIDER_MAX_CARDS,
        copy_docx_into_site=COPY_DOCX_INTO_EACH_SITE,
        replace_duplicate_post_id=REPLACE_DUPLICATE_POST_ID,
        update_home_floating=UPDATE_HOME_FLOATING,
        update_home_slider=UPDATE_HOME_SLIDER,
    )
    print("Weekly post sync completed for these site roots:")
    for item in weekly_results:
        print("
SITE:", item["site_root"])
        print("  post ID:", item["post_id"])
        print("  title:", item["title"])
        print("  ongoing-work:", item["ongoing_work"])
        print("  index:", item["index_html"])
        print("  full post:", item["post_html"])
        if item["notes"]:
            print("  notes:", item["notes"])
else:
    print("Weekly post sync is disabled.")

In [ ]:
if RUN_ARTICLE_SYNC:
    article_source_site = (WORKSPACE_ROOT / ARTICLE_SOURCE_REPO_NAME / ARTICLE_SOURCE_SITE_RELATIVE).resolve()
    manifest_entries = read_manifest(ARTICLE_MANIFEST_PATH)
    article_results = sync_article_manifest_to_sites(
        source_site_root=article_source_site,
        target_site_roots=SITE_ROOTS,
        manifest_entries=manifest_entries,
    )
    print("Article/page sync copied these files:")
    for item in article_results:
        print("
PATH:", item["relative_path"])
        print("  source:", item["source"])
        print("  target:", item["target"])
else:
    print("Article/page sync is disabled.")

## Notes

### Weekly post mode
Use this when you have one filled DOCX and want every codebase copy updated together.

### Article/page sync mode
Use this when you already edited a static page in one source site and want that finished result mirrored everywhere else.

### Safe habit
Run one workflow at a time, review the changed files, then commit.